In [1]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-openai
!pip install -U langchain-text-splitters
!pip install -U chromadb
!pip install -U pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 14.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.0
    Uninstalling langgraph-checkpoint-4.0.0:
      Successfully uninstalled langgraph-checkpoint-4.0.0
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt

In [4]:
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from langchain_community.vectorstores import Chroma

os.environ["OPENAI_API_KEY"] = ""


# STEP 1: LOAD PDF
loader = PyPDFLoader("/kaggle/input/datasets/ashupandey1620/research-paper-draft/MLLM4EDA_v1 (1).pdf")

documents = loader.load()


# STEP 2: SPLIT DOCUMENTS
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)


# STEP 3: EMBEDDINGS
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# STEP 4: VECTOR STORE
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)


# STEP 5: RETRIEVER
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


# STEP 6: LLM
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# STEP 7: QUERY
query = "Summarize the document"


# STEP 8: RETRIEVE DOCUMENTS
retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)


# STEP 9: CREATE PROMPT
prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""


# STEP 10: GENERATE RESPONSE
response = llm.invoke(prompt)

print(response.content)

The document discusses combining vision and text features using an additive fusion mechanism, particularly in the context of EDA workflows that involve both text and diagrams. It compares different models, noting that GPT-4o generates coherent but often shallow and nonspecific outputs, while the base model struggles with reliability and relevance. In contrast, a fine-tuned model effectively adapts to the domain, producing accurate, contextually appropriate, and technically detailed responses aligned with reference outputs.
